# Transformers
### Author: Krzysztof Chmielewski

## Introduction
## Self-Attention
Consider a sentence: *"The cat sat on the mat because **it** was tired."*
When the model reads the word "**it**", it must decide:

> Does "**it**" refer to the *cat* or to the *mat*?

Self-attention solves this by letting every word look at every other word and decide what matters.

Transformers operate using self-attention mechanism that allows them to recognize which other tokens in a given sequence are important for considered token. For every token attention is computed:

$$
Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}}) \cdot V
$$

Where:
- $Q$ is query (what token is looking for)
- $K$ is key (what token contains)
- $V$ is value (what token provides)

### **Let's break it down**

Suppose:
- Sequence length: $n$
- Embedding dimension: $d$

Input matrix: $$ X \in \R^{n \times d} $$ 

Where each row is one token embedding. We create three weight matrices:
$$ W_Q, W_K, W_V \in \R^{d \times d_k} $$ and then compute: $$ Q = XW_Q $$ $$ K = XW_K$$ $$ V = XW_V $$
Now $$ Q,K,V \in \R^{n \times d_k} $$

### Score and dot product
Given $ Q,K,V $ when a token i-th token wants context it uses its query ($Q_i$) and compares with every key ($K_j$). Attention score is computed as **dot product**: $$ score_{ij} = Q_i \cdot K_j $$

Dot products measures similarity:
- Large positive - vectors aligned (similar)
- Near zero - vectors almost orthogonal (unrelated)
- Negative - vectors point in opposite directions (opposite)

If "**it**" in considered previously sentence is similar to "*cat*" in Q-K space, their dot product will be high.

Matrix $QK^T$ tells us
> For each token $i$ (row) how much attention it gives to each token $j$ (column)

### Dividing by $\sqrt{d_k}$
This is for normalization purpose $\rarr$ as dimension $d_k$ increases the dot product of $QK^T$ matrix grows in magnitude which results in softmax saturation and gradient vanish. Without this operation training becomes unstable.

### Softmax
Now we have apply softmax: $$ softmax(\frac{QK^T}{\sqrt{d_k}}) $$ Softmax turns scores into probabilities where row sums to 1. So each token distributes 100% of its attention across all tokens.

### Weighted sum of values
Now we multiply this probabilities by values matrix $V = XW_V$ and get final form:

$$
Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}}) \cdot V
$$

Why is that? - The attention matrix tells us *how much to listen to each token* and values contain the actual information.

Final representation of i-th token is as follows:
$$
t_i = \sum_{j=1}^n \alpha_{ij}V_j
$$
Where $\alpha_{ij}$ is attention weight (how much attention i-th token gives to j-th token).


## Multi-head attention
Given token (for example a *word* as in previous considered sentence) can depend context-wise on many different other tokens, not only one. Here we give transformers the ability to recognize that by using many **attention heads**.

Single-head attention is powerful. Multi-head attention is about learning multiple different relational spaces simultaneously. It computes attention as previously
$$
Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}}) \cdot V
$$

but **h times**. For i-th head: $$ Q_i = XW_Q^{(i)} $$ $$ K_i = XW_K^{(i)} $$ $$ V_i = XW_V^{(i)} $$
Where $$ W_Q^{(i)}, W_K^{(i)}, W_V^{(i)} \in \R^{d_{model} \times d_k} $$ Each head has its own parameter matrices. So we compute attention per head:
$$
head_i = softmax(\frac{Q_iK_i^T}{\sqrt{d_k}}) \cdot V_i
$$
And each $ head_i \in \R^{n \times d_k} $

### Head concatenation

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, embed_size):
        super(SelfAttention, self).__init__()

        self.embed_size = embed_size

        # Linear layers to create Q, K, V
        self.values = nn.Linear(embed_size, embed_size)
        self.keys = nn.Linear(embed_size, embed_size)
        self.queries = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len, embed_size)

        values = self.values(x)
        keys = self.keys(x)
        queries = self.queries(x)

        # Compute attention scores and scale them
        energy = torch.matmul(queries, keys.transpose(-2, -1)) / (self.embed_size ** 0.5)

        # Softmax
        attention = torch.softmax(energy, dim=-1)

        # Multiply by values
        out = torch.matmul(attention, values)

        return out
